# 04 - Privacy Analysis

This notebook explores the privacy side of Study II: pairwise linkability robustness, hard negatives, privacy transformations, PCA/random projections, fitted transformation parameters, and operational-style gallery linkage.

The goal is to distinguish three things:
- sampled pairwise separability;
- robustness of that separability under harder pairs;
- lower-prevalence synthetic gallery behavior.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import OUTPUTS_FIGURES_DIR, OUTPUTS_TABLES_DIR

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (8, 4)


## 1. Linkability Robustness

The first robustness check keeps the pairwise task but makes it less trivial: larger temporal gaps, fewer positive pairs per patient, hard negatives, and simple distance baselines.


In [ ]:
linkability_robustness_df = pd.read_csv(OUTPUTS_TABLES_DIR / "linkability_robustness_summary.csv")
linkability_robustness_df


In [ ]:
robustness_pivot = linkability_robustness_df.pivot_table(index="model", columns="setting", values="roc_auc")
display(robustness_pivot)

robustness_pivot.plot.bar(figsize=(9, 4), color=["#457b9d", "#e76f51", "#2a9d8f"])
plt.title("Linkability robustness: ROC-AUC by setting")
plt.xlabel("Model / baseline")
plt.ylabel("ROC-AUC")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 2. Temporal Gap and Hard Negatives Sensitivity

This check varies the temporal separation required for positive pairs and compares random negatives with hard negatives. Hard negatives are the more informative stress test.


In [ ]:
linkability_sensitivity_df = pd.read_csv(OUTPUTS_TABLES_DIR / "linkability_sensitivity_gap_hardneg.csv")
linkability_sensitivity_df


In [ ]:
xgb_sensitivity = linkability_sensitivity_df[linkability_sensitivity_df["model"] == "XGBoost"].copy()
for strategy, group in xgb_sensitivity.groupby("negative_strategy"):
    plt.plot(group["min_segment_gap"], group["roc_auc"], marker="o", label=strategy)
plt.title("XGBoost linkability sensitivity to temporal gap and negative strategy")
plt.xlabel("Minimum positive-pair segment gap")
plt.ylabel("ROC-AUC")
plt.legend(title="Negative strategy")
plt.tight_layout()
plt.show()


## 3. Split-Aware Privacy Transformations

These results come from the final transformation pipeline: transformations are fitted on train only, applied to train/test, and pairs are built after transformation. The summary is multi-seed across `[42, 123, 456]`.


In [ ]:
privacy_transformations_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transformations_summary.csv")
privacy_transform_full_metrics_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transform_full_final_metrics.csv")
privacy_transform_full_summary_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transform_full_final_summary.csv")

privacy_transformations_df.sort_values("delta_linkability_roc_auc")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    privacy_transformations_df["linkability_roc_auc"],
    privacy_transformations_df["utility_f1"],
    s=80,
    color="#4c78a8",
)
for _, row in privacy_transformations_df.iterrows():
    ax.annotate(row["transform_name"], (row["linkability_roc_auc"], row["utility_f1"]), xytext=(5, 4), textcoords="offset points")
ax.set_xlabel("Linkability ROC-AUC")
ax.set_ylabel("Utility F1")
ax.set_title("Final split-aware transformation trade-off")
plt.tight_layout()
plt.show()


In [ ]:
selected_transforms = ["identity", "pca_120", "rp_120", "quant_dec0", "winsor_05_95", "noise_010"]
per_seed_view = privacy_transform_full_metrics_df[
    privacy_transform_full_metrics_df["transform_name"].isin(selected_transforms)
][
    [
        "seed",
        "transform_name",
        "utility_f1",
        "utility_balanced_accuracy",
        "utility_roc_auc",
        "linkability_roc_auc",
        "linkability_pr_auc",
    ]
].sort_values(["transform_name", "seed"])
per_seed_view


## 4. Selective Privacy Transformations

This experiment uses the feature-overlap analysis to apply stronger protection only to features that are more linkability-dominant, while preserving utility-dominant features. The current full-data result is a seed-42 exploratory check, not yet a multi-seed final estimate.

The main top-80 run can be reproduced with:

```powershell
.\venv\Scripts\python.exe src\run_selective_privacy_transform_study.py --max-chunks 0 --seeds 42 --run-name selective_privacy_transform_top80_seed42 --candidates-path .\outputs\tables\feature_group_importance_full_top80_customization_candidates.csv --top-per-type 80 --pca-components 10 20 30 --link-max-pairs 2000 --experiments identity link_pca_10 link_pca_20 link_pca_30
```


In [ ]:
selective_summary_path = OUTPUTS_TABLES_DIR / "selective_privacy_transform_all_seed42_summary.csv"
selective_policy_path = OUTPUTS_TABLES_DIR / "selective_privacy_transform_top80_seed42_feature_policy.csv"
selective_protocol_path = OUTPUTS_TABLES_DIR / "selective_privacy_transform_top80_seed42_protocol.json"

selective_transform_summary_df = pd.read_csv(selective_summary_path)
selective_feature_policy_df = pd.read_csv(selective_policy_path)
selective_protocol = json.loads(selective_protocol_path.read_text(encoding="utf-8"))

selective_cols = [
    "experiment_family",
    "transform_name",
    "n_output_features",
    "n_linkability_dominant_features",
    "n_shared_high_features",
    "n_preserved_utility_dominant_features",
    "utility_f1_mean",
    "utility_balanced_accuracy_mean",
    "utility_roc_auc_mean",
    "linkability_roc_auc_mean",
    "linkability_pr_auc_mean",
    "pca_explained_variance_ratio_sum_mean",
    "delta_utility_f1",
    "delta_linkability_roc_auc",
]
display(selective_transform_summary_df[selective_cols].sort_values("linkability_roc_auc_mean").round(4))

policy_counts = pd.DataFrame([
    {"policy_group": group_name, "n_features": len(features)}
    for group_name, features in selective_protocol["feature_policy"].items()
])
display(policy_counts)

policy_examples = pd.DataFrame([
    {"policy_group": group_name, "example_features": ", ".join(features[:8])}
    for group_name, features in selective_protocol["feature_policy"].items()
])
display(policy_examples)


In [ ]:
OUTPUTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plot_df = selective_transform_summary_df.copy()
plot_df["transform_label"] = plot_df["transform_name"].str.replace("link_", "", regex=False)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(plot_df["transform_label"], plot_df["utility_f1_mean"], marker="o", color="#3A6EA5", label="Utility F1")
ax2.plot(plot_df["transform_label"], plot_df["linkability_roc_auc_mean"], marker="s", color="#C44E52", label="Linkability ROC-AUC")
ax1.set_ylabel("Utility F1", color="#3A6EA5")
ax2.set_ylabel("Linkability ROC-AUC", color="#C44E52")
ax1.set_xlabel("Selective transformation")
ax1.set_title("Aggressive selective privacy frontier")
ax1.tick_params(axis="x", rotation=25)
ax1.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUTS_FIGURES_DIR / "selective_privacy_transform_all_seed42_frontier.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Supervised Bottleneck Encoder

A stronger mitigation strategy is to learn a compact representation supervised only by the utility task. The encoder is trained on the train split to predict `Atrial` vs `Non-Atrial`; the final hidden bottleneck layer is then used as the feature representation for both utility and linkability evaluation.

This asks whether a representation optimized for utility can discard part of the information that makes segments linkable.


In [ ]:
encoder_summary_df = pd.read_csv(OUTPUTS_TABLES_DIR / "supervised_bottleneck_encoder_allseeds_summary.csv")
encoder_metrics_df = pd.read_csv(OUTPUTS_TABLES_DIR / "supervised_bottleneck_encoder_allseeds_metrics.csv")
encoder_protocol = json.loads((OUTPUTS_TABLES_DIR / "supervised_bottleneck_encoder_seed42_protocol.json").read_text(encoding="utf-8"))

encoder_cols = [
    "bottleneck_dim",
    "n_embedding_features",
    "utility_f1_mean",
    "utility_f1_std",
    "utility_balanced_accuracy_mean",
    "utility_roc_auc_mean",
    "linkability_roc_auc_mean",
    "linkability_roc_auc_std",
    "linkability_pr_auc_mean",
]
display(encoder_summary_df[encoder_cols].round(4))

display(
    encoder_metrics_df[[
        "seed",
        "bottleneck_dim",
        "encoder_train_segments",
        "encoder_n_iter",
        "utility_f1",
        "utility_balanced_accuracy",
        "linkability_roc_auc",
        "linkability_pr_auc",
    ]].sort_values(["bottleneck_dim", "seed"]).round(4)
)


In [ ]:
frontier_with_encoder_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_frontier_with_supervised_encoder_summary.csv")

fig, ax = plt.subplots(figsize=(8, 5))
color_map = {
    "top80_selective": "#9A8C98",
    "preserve_utility80_rest": "#3A6EA5",
    "preserve_utility50_rest": "#C44E52",
    "preserve_utility30_rest": "#2A9D8F",
    "supervised_bottleneck_encoder": "#111111",
}
for family, group in frontier_with_encoder_df.groupby("method_family"):
    ax.scatter(
        group["linkability_roc_auc_mean"],
        group["utility_f1_mean"],
        s=85,
        alpha=0.88,
        label=family,
        color=color_map.get(family),
    )
for _, row in frontier_with_encoder_df.iterrows():
    if row["method_family"] == "supervised_bottleneck_encoder" or row["label"] == "identity":
        ax.annotate(row["label"], (row["linkability_roc_auc_mean"], row["utility_f1_mean"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Linkability ROC-AUC")
ax.set_ylabel("Utility F1")
ax.set_title("Privacy-utility frontier with supervised encoder")
ax.grid(alpha=0.25)
ax.legend(title="Method family", fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUTS_FIGURES_DIR / "privacy_frontier_with_supervised_encoder.png", dpi=200, bbox_inches="tight")
plt.show()


### Encoder + DP-Calibrated Noise

Because direct DP-style noise on the full 208-feature vector is very destructive, this experiment first learns an 8-dimensional supervised bottleneck representation and then applies Gaussian noise calibrated by epsilon, delta, and L2 clipping in the embedding space. This is DP-calibrated embedding perturbation, not DP-SGD training.


In [ ]:
encoder_dp_df = pd.read_csv(OUTPUTS_TABLES_DIR / "encoder_dp_dim8_seed42_compact_summary.csv")
encoder_dp_df[[
    "transform_name",
    "dp_epsilon",
    "dp_clip_norm",
    "dp_noise_sigma",
    "utility_f1_mean",
    "utility_balanced_accuracy_mean",
    "utility_roc_auc_mean",
    "linkability_roc_auc_mean",
    "linkability_pr_auc_mean",
]]


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2))
for clip, group in encoder_dp_df.groupby("dp_clip_norm", dropna=False):
    label = "encoder identity" if pd.isna(clip) else f"DP clip={clip:g}"
    ax.scatter(group["linkability_roc_auc_mean"], group["utility_balanced_accuracy_mean"], s=80, label=label)
    for _, row in group.iterrows():
        text = "identity" if pd.isna(row["dp_epsilon"]) else f"eps={row["dp_epsilon"]:g}"
        ax.annotate(text, (row["linkability_roc_auc_mean"], row["utility_balanced_accuracy_mean"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.set_xlabel("Linkability ROC-AUC")
ax.set_ylabel("Utility balanced accuracy")
ax.set_title("Encoder + DP trade-off, seed 42")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
figure_path = OUTPUTS_FIGURES_DIR / "encoder_dp_dim8_seed42_tradeoff.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
figure_path


### Operational Linkage for Encoder + DP

The best encoder+DP configurations are also evaluated with the synthetic-gallery protocol. This checks whether the DP-calibrated embedding perturbation lowers low-prevalence PR-AUC, Recall@1 and MRR beyond the balanced pairwise ROC-AUC setting.


In [ ]:
encoder_dp_operational_df = pd.read_csv(OUTPUTS_TABLES_DIR / "operational_linkage_encoder_dp_comparison.csv")
encoder_dp_operational_df[[
    "transform_label",
    "gallery_negatives",
    "positive_prevalence",
    "dp_epsilon",
    "dp_clip_norm",
    "utility_balanced_accuracy",
    "link_operational_roc_auc",
    "link_operational_pr_auc",
    "recall_at_1",
    "recall_at_5",
    "mrr",
]]


In [ ]:
plot_df = encoder_dp_operational_df[encoder_dp_operational_df["gallery_negatives"].eq(999)].copy()
order = [
    "raw 208 features",
    "encoder dim8",
    "encoder+DP eps100 c2",
    "encoder+DP eps50 c2",
    "encoder+DP eps50 c4",
    "encoder+DP eps20 c4",
]
plot_df["transform_label"] = pd.Categorical(plot_df["transform_label"], categories=order, ordered=True)
plot_df = plot_df.sort_values("transform_label")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].bar(plot_df["transform_label"].astype(str), plot_df["link_operational_pr_auc"], color="#457b9d")
axes[0].set_title("Operational PR-AUC, 0.1% prevalence")
axes[0].set_ylabel("PR-AUC")
axes[0].set_ylim(0, 1.02)
axes[0].tick_params(axis="x", rotation=25)
axes[0].grid(axis="y", alpha=0.25)
axes[1].bar(plot_df["transform_label"].astype(str), plot_df["recall_at_1"], color="#e76f51")
axes[1].set_title("Recall@1, 0.1% prevalence")
axes[1].set_ylabel("Recall@1")
axes[1].set_ylim(0, 1.02)
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
figure_path = OUTPUTS_FIGURES_DIR / "encoder_dp_operational_dim8_seed42.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
figure_path


### Representation Dimension Benchmark

To verify that the supervised encoder is not simply benefiting from lower dimensionality, this benchmark compares it against PCA, random projection, and top-k utility-ranked features using the same representation dimensions.


In [ ]:
repr_dim_summary_df = pd.read_csv(OUTPUTS_TABLES_DIR / "representation_dimension_comparison_seed42_full_summary.csv")

repr_cols = [
    "method",
    "dimension",
    "utility_f1_mean",
    "utility_balanced_accuracy_mean",
    "utility_roc_auc_mean",
    "linkability_roc_auc_mean",
    "linkability_pr_auc_mean",
]
display(repr_dim_summary_df[repr_cols].sort_values(["method", "dimension"]).round(4))

best_by_method_df = (
    repr_dim_summary_df[repr_dim_summary_df["method"] != "identity"]
    .sort_values(["utility_f1_mean", "linkability_roc_auc_mean"], ascending=[False, True])
    .groupby("method", as_index=False)
    .head(1)
)
display(best_by_method_df[repr_cols].round(4))


In [ ]:
fig, ax1 = plt.subplots(figsize=(8.5, 5))
ax2 = ax1.twinx()
method_colors = {
    "supervised_encoder": "#111111",
    "pca": "#3A6EA5",
    "random_projection": "#C44E52",
    "topk_utility": "#2A9D8F",
}
curve_df = repr_dim_summary_df[repr_dim_summary_df["method"] != "identity"].copy()
for method, group in curve_df.groupby("method"):
    group = group.sort_values("dimension")
    ax1.plot(group["dimension"], group["utility_f1_mean"], marker="o", color=method_colors.get(method), label=f"{method} utility F1")
    ax2.plot(group["dimension"], group["linkability_roc_auc_mean"], marker="s", linestyle="--", color=method_colors.get(method), alpha=0.75, label=f"{method} link ROC-AUC")
ax1.set_xscale("log", base=2)
ax1.set_xlabel("Representation dimension")
ax1.set_ylabel("Utility F1")
ax2.set_ylabel("Linkability ROC-AUC")
ax1.set_title("Representation dimension vs utility/linkability")
ax1.grid(alpha=0.25)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="center right")
fig.tight_layout()
fig.savefig(OUTPUTS_FIGURES_DIR / "representation_dimension_comparison_seed42_full.png", dpi=200, bbox_inches="tight")
plt.show()


## 6. Transformation Protocol and Fitted Parameters

The fitted parameters are stored separately so the transformations are reproducible without relying only on prose. The parameter table includes imputer medians, scaler statistics, winsor bounds, quantization decimals and noise parameters where relevant.


In [ ]:
protocol = json.loads((OUTPUTS_TABLES_DIR / "privacy_transform_full_final_protocol.json").read_text(encoding="utf-8"))
protocol_summary = pd.DataFrame(
    [
        {"field": "dataset", "value": protocol["dataset_dir"]},
        {"field": "loaded_segments", "value": protocol["loaded_segments"]},
        {"field": "seeds", "value": protocol["seeds"]},
        {"field": "transformation_fit_unit", "value": protocol["transformation_protocol"]["fit_unit"]},
        {"field": "pair_order", "value": protocol["transformation_protocol"]["pair_order"]},
        {"field": "linkability_pairs", "value": protocol["linkability_protocol"]["max_pairs"]},
        {"field": "min_segment_gap", "value": protocol["linkability_protocol"]["min_segment_gap"]},
        {"field": "pair_prevalence", "value": protocol["linkability_protocol"]["pair_prevalence"]},
    ]
)
protocol_summary


In [ ]:
fitted_params_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transform_fitted_parameters.csv")

winsor_bounds_example = fitted_params_df[
    (fitted_params_df["transform_name"] == "winsor_05_95")
    & (fitted_params_df["seed"] == 42)
][["feature", "imputer_median", "winsor_lower_bound", "winsor_upper_bound"]].head(12)

noise_params = fitted_params_df[fitted_params_df["method"] == "noise"][[
    "transform_name", "seed", "noise_std", "noise_random_state", "n_output_features"
]].drop_duplicates().sort_values(["transform_name", "seed"])

display(winsor_bounds_example)
display(noise_params)


## 7. PCA Components and Explained Variance

Two PCA views are useful: the old component-count sweep and the final fitted PCA(120) explained-variance profile.


In [ ]:
pca_components_df = pd.read_csv(OUTPUTS_TABLES_DIR / "pca_components_tradeoff_curve.csv")
pca_components_df


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(pca_components_df["n_components"], pca_components_df["utility_f1"], marker="o", color="#457b9d", label="Utility F1")
ax2.plot(pca_components_df["n_components"], pca_components_df["linkability_roc_auc"], marker="s", color="#d62828", label="Linkability ROC-AUC")
ax1.set_xlabel("Number of PCA components")
ax1.set_ylabel("Utility F1", color="#457b9d")
ax2.set_ylabel("Linkability ROC-AUC", color="#d62828")
ax1.set_title("PCA components vs utility/linkability")
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
plt.tight_layout()
plt.show()


In [ ]:
pca_explained_df = pd.read_csv(OUTPUTS_TABLES_DIR / "privacy_transform_pca_explained_variance.csv")
pca_curve = (
    pca_explained_df.sort_values(["seed", "component"])
    .assign(cumulative_variance=lambda df: df.groupby("seed")["explained_variance_ratio"].cumsum())
)

for seed, group in pca_curve.groupby("seed"):
    plt.plot(group["component"] + 1, group["cumulative_variance"], label=f"seed {seed}")
plt.title("Cumulative explained variance for fitted PCA(120)")
plt.xlabel("Component")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.tight_layout()
plt.show()

pca_curve.groupby("seed")["explained_variance_ratio"].sum().reset_index(name="total_explained_variance")


## 8. Operational-Style Linkage Metrics

Balanced pairwise ROC-AUC is not an operational re-identification risk by itself. This synthetic gallery check lowers positive prevalence and reports PR-AUC, TPR at low FPR, Recall@k and MRR.


In [ ]:
operational_linkage_summary_df = pd.read_csv(OUTPUTS_TABLES_DIR / "operational_linkage_final_summary.csv")
operational_linkage_metrics_df = pd.read_csv(OUTPUTS_TABLES_DIR / "operational_linkage_final_metrics.csv")

operational_linkage_summary_df[[
    "gallery_negatives",
    "positive_prevalence_mean",
    "roc_auc_mean",
    "pr_auc_mean",
    "tpr_at_fpr_0.001_mean",
    "recall_at_1_mean",
    "recall_at_5_mean",
    "mrr_mean",
]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(operational_linkage_summary_df["gallery_negatives"].astype(str), operational_linkage_summary_df["pr_auc_mean"], color="#457b9d")
axes[0].set_title("Operational linkage PR-AUC")
axes[0].set_xlabel("Negative candidates per query")
axes[0].set_ylabel("PR-AUC")
axes[1].bar(operational_linkage_summary_df["gallery_negatives"].astype(str), operational_linkage_summary_df["recall_at_1_mean"], color="#e76f51")
axes[1].set_title("Recall@1")
axes[1].set_xlabel("Negative candidates per query")
axes[1].set_ylabel("Recall@1")
plt.tight_layout()
plt.show()


### Encoder Operational Linkage Comparison

The supervised bottleneck encoder is also evaluated under the same synthetic-gallery protocol. This comparison is useful because low-prevalence PR-AUC and Recall@k are closer to an operational linkage setting than balanced pairwise ROC-AUC.


In [ ]:
operational_repr_comparison_df = pd.read_csv(OUTPUTS_TABLES_DIR / "operational_linkage_representation_comparison.csv")
operational_repr_comparison_df[[
    "representation",
    "gallery_negatives",
    "positive_prevalence_mean",
    "utility_f1_mean",
    "utility_balanced_accuracy_mean",
    "link_operational_roc_auc_mean",
    "link_operational_pr_auc_mean",
    "tpr_at_fpr_0.001_mean",
    "recall_at_1_mean",
    "recall_at_5_mean",
    "mrr_mean",
]]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
plot_df = operational_repr_comparison_df.copy()
plot_df["gallery_label"] = plot_df["gallery_negatives"].map({99: "1% prevalence", 999: "0.1% prevalence"})
for ax, metric, title in [
    (axes[0], "link_operational_pr_auc_mean", "Operational PR-AUC"),
    (axes[1], "recall_at_1_mean", "Recall@1"),
]:
    pivot = plot_df.pivot(index="representation", columns="gallery_label", values=metric)
    pivot.plot(kind="bar", ax=ax, width=0.75)
    ax.set_title(title)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
figure_path = OUTPUTS_FIGURES_DIR / "operational_linkage_representation_comparison.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
figure_path


## Main Takeaways

- Pairwise linkability remains very high across robustness checks, including harder temporal gaps and hard negatives.
- Global PCA/random projection reduce linkability more than simple noise, quantization, scaling, or winsorization, but utility also declines slightly.
- Selective PCA/noise over only linkability-dominant features preserves utility but does not reduce linkability enough, suggesting identifiable information is distributed across feature families.
- More aggressive preserve-utility/rest-protected projections reduce linkability further, but the utility cost becomes visible.
- The supervised bottleneck encoder gives the strongest current privacy-utility frontier: utility improves relative to the segment baseline while linkability ROC-AUC drops substantially.
- Encoder + DP-calibrated noise gives an explicit privacy-budget curve; `eps50/clip2` keeps balanced accuracy around 0.83 while reducing linkability ROC-AUC to around 0.79, and `eps50/clip4` reduces linkability further with a larger utility cost.
- Under the operational gallery protocol, encoder+DP further reduces low-prevalence PR-AUC and Recall@1; at 0.1% prevalence, `eps50/clip2` reduces Recall@1 from 0.062 to 0.017 while keeping balanced accuracy around 0.829.
- Operational-style gallery metrics for the raw 208-feature representation remain very high, but the supervised bottleneck encoder strongly lowers low-prevalence PR-AUC and Recall@1.
- These experiments are noise/projection/encoder privacy checks, not formal differential privacy, because no explicit epsilon/delta accounting or sensitivity-calibrated DP mechanism is used.
